# 03: Bias removal and train/val/test splits (Step 5)

Builds the processed dataset used by every experiment:
1. Drop BigGAN (128 px, too small for a 224 px crop), grayscale real images, and real images smaller than 224 px.
2. Estimate the JPEG quality of every real image.
3. Split: HF train into our train/val (85/15, per source), HF validation into our test.
4. Balance real and fake within each split.
5. Crop every image to 224 x 224 at a 16-pixel-aligned offset; fake images get one JPEG compression with a quality drawn from the real images' quality distribution.
6. Save arrays and metadata to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets

import io
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from datasets import load_dataset, Image as HFImage

ROOT = "/content/drive/MyDrive/minor-project"
SIZE = 224

ds = load_dataset("TheKernel01/Tiny-GenImage", cache_dir="/content/hf_cache")
parts = {split: ds[split].cast_column("image", HFImage(decode=False)) for split in ds}

inventory = pd.read_csv(Path(ROOT, "results", "03_tiny_genimage_inventory.csv"))
inventory = inventory.rename(columns={"split": "hf_split", "index": "hf_index"})
print(len(inventory), "images in inventory")

## Helper functions

In [ ]:
# Standard JPEG luminance quantization table (JPEG standard, Annex K)
STD_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
])


def estimate_quality(img):
    luma = np.array(img.quantization[0])
    best_quality, best_error = 0, np.inf
    for quality in range(1, 101):
        scale = 5000 // quality if quality < 50 else 200 - 2 * quality
        table = np.clip((STD_LUMA * scale + 50) // 100, 1, 255)
        error = np.abs(table - luma).sum()
        if error < best_error:
            best_quality, best_error = quality, error
    return best_quality, int(best_error)


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))


def jpeg_roundtrip(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=int(quality))
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

## JPEG quality of the real images

In [ ]:
inventory["jpeg_quality"] = np.nan
inventory["table_error"] = np.nan
is_real = inventory["label"] == "real"

for row in inventory[is_real].itertuples():
    raw = parts[row.hf_split][row.hf_index]["image"]["bytes"]
    quality, error = estimate_quality(Image.open(io.BytesIO(raw)))
    inventory.loc[row.Index, ["jpeg_quality", "table_error"]] = [quality, error]

real_quality = inventory.loc[is_real, ["hf_split", "hf_index", "jpeg_quality", "table_error"]]
real_quality.to_csv(Path(ROOT, "results", "04_real_jpeg_quality.csv"), index=False)

print("Most common JPEG qualities among real images:")
display(real_quality["jpeg_quality"].value_counts().head(10))
print(f"Real images using the standard JPEG tables exactly: {(real_quality['table_error'] == 0).mean():.1%}")

## Filter, split and balance

In [ ]:
keep = (
    (inventory["generator"] != "BigGAN")
    & (inventory["mode"] != "L")
    & (inventory[["width", "height"]].min(axis=1) >= SIZE)
)
data = inventory[keep].copy()
print(f"Kept {len(data)} of {len(inventory)} images")

data["our_split"] = "test"
from_train = data["hf_split"] == "train"
data.loc[from_train, "our_split"] = "train"
val_rows = data[from_train].groupby("generator", group_keys=False).sample(frac=0.15, random_state=42)
data.loc[val_rows.index, "our_split"] = "val"

balanced = []
for split_name, group in data.groupby("our_split"):
    fakes = group[group["label"] == "fake"]
    reals = group[group["label"] == "real"]
    n = min(len(fakes), len(reals))
    balanced.append(pd.concat([reals.sample(n=n, random_state=42), fakes.sample(n=n, random_state=42)]))
data = pd.concat(balanced)

real_train_quality = data.loc[(data["label"] == "real") & (data["our_split"] == "train"), "jpeg_quality"].to_numpy()
rng = np.random.default_rng(42)
is_fake = data["label"] == "fake"
data.loc[is_fake, "jpeg_quality"] = rng.choice(real_train_quality, size=is_fake.sum())
data["jpeg_quality"] = data["jpeg_quality"].astype(int)

data = data.sample(frac=1, random_state=42).reset_index(drop=True)
display(pd.crosstab(data["generator"], data["our_split"], margins=True))

## Build the processed arrays (about 10 minutes)

In [ ]:
OUT = Path("/content/processed")
OUT.mkdir(exist_ok=True)

for split_name in ["train", "val", "test"]:
    rows = data[data["our_split"] == split_name].reset_index(drop=True)
    images = np.lib.format.open_memmap(
        OUT / f"{split_name}_images.npy", mode="w+", dtype=np.uint8, shape=(len(rows), SIZE, SIZE, 3)
    )
    for i, row in enumerate(rows.itertuples()):
        raw = parts[row.hf_split][row.hf_index]["image"]["bytes"]
        img = Image.open(io.BytesIO(raw)).convert("RGB")
        img = center_crop_aligned(img)
        if row.label == "fake":
            img = jpeg_roundtrip(img, row.jpeg_quality)
        images[i] = np.asarray(img)
        if i % 2000 == 0:
            print(f"{split_name}: {i}/{len(rows)}")
    images.flush()
    del images
    rows.to_csv(OUT / f"{split_name}_meta.csv", index=False)
    print(f"{split_name}: done, {len(rows)} images")

## Checks

In [ ]:
print("JPEG quality by split and label (should match between real and fake):")
display(data.groupby(["our_split", "label"])["jpeg_quality"].describe()[["count", "mean", "50%", "min", "max"]].round(1))
data.to_csv(Path(ROOT, "results", "04_processed_metadata.csv"), index=False)

In [ ]:
import matplotlib.pyplot as plt

images = np.load(OUT / "test_images.npy", mmap_mode="r")
meta = pd.read_csv(OUT / "test_meta.csv")
sources = ["Real", "ADM", "GLIDE", "Midjourney", "SD15", "VQDM", "Wukong"]

fig, axes = plt.subplots(3, len(sources), figsize=(2.2 * len(sources), 7))
for col, source in enumerate(sources):
    picks = meta.index[meta["generator"] == source][:3]
    for r, idx in enumerate(picks):
        axes[r, col].imshow(images[idx])
        axes[r, col].axis("off")
    axes[0, col].set_title(source)
plt.tight_layout()
plt.savefig(Path(ROOT, "results", "04_processed_samples.png"), dpi=100)
plt.show()

## Save to Drive

In [ ]:
DRIVE_OUT = Path(ROOT, "processed")
DRIVE_OUT.mkdir(exist_ok=True)
for f in sorted(OUT.iterdir()):
    shutil.copy(f, DRIVE_OUT / f.name)
    print(f"copied {f.name} ({f.stat().st_size / 1e9:.2f} GB)")